# Human + Multi-Agent Organizations

**Level:** Advanced · **Time:** 90 min

In this comprehensive notebook, we simulate the design of a mixed Human/AI organization.

We will cover 4 distinct patterns:
1. **The Vague Handoff vs The Work Order:** Why vague natural language delegation fails, and JSON contracts succeed.
2. **Context Isolation:** How a Manager Agent protects specialist agents from hallucinating by restricting their data feeds.
3. **The Escalation Path:** Forcing an agent to surrender rather than hallucinate when data is missing.
4. **Human-in-the-Loop (HITL) Execution:** Pausing an agent graph for a cryptographic human signature before a destructive action.

---
## Pattern 1: The Vague Handoff vs The Work Order

Delegating to an agent using vague language is an anti-pattern. The agent will guess your intentions and likely violate policy. You must use a typed Work Order.

In [1]:
def mock_agent_execution(instruction: dict):
    if instruction.get("type") == "vague_natural_language":
        print("[Agent] Received vague prompt: 'Fix the checkout.'")
        print("[Agent] I see fraud checks are failing. I will disable the fraud system to fix the checkout.")
        return "🚨 DISASTER: Security disabled."
    
    elif instruction.get("type") == "work_order":
        print("[Agent] Received strict JSON Work Order.")
        print(f"  > Objective: {instruction['objective']}")
        print(f"  > No-Go Actions: {instruction['no_go_actions']}")
        print("[Agent] I will analyze the logs without modifying the fraud system.")
        return "✅ SUCCESS: Logs analyzed, root cause identified safely."

print("--- Scenario A: The Vague Handoff ---")
mock_agent_execution({"type": "vague_natural_language", "prompt": "Fix the checkout."})

print("\n--- Scenario B: The Work Order ---")
work_order = {
    "type": "work_order",
    "objective": "Identify the root cause of the EU checkout failure.",
    "no_go_actions": ["Do not modify payment gateways", "Do not disable security checks"],
    "expected_artifact": "RootCauseAnalysisJSON"
}
mock_agent_execution(work_order)


--- Scenario A: The Vague Handoff ---
[Agent] Received vague prompt: 'Fix the checkout.'
[Agent] I see fraud checks are failing. I will disable the fraud system to fix the checkout.

--- Scenario B: The Work Order ---
[Agent] Received strict JSON Work Order.
  > Objective: Identify the root cause of the EU checkout failure.
  > No-Go Actions: ['Do not modify payment gateways', 'Do not disable security checks']
[Agent] I will analyze the logs without modifying the fraud system.


'✅ SUCCESS: Logs analyzed, root cause identified safely.'

---
## Pattern 2: Context Isolation

A Manager Agent's primary job is to isolate context. If you feed all incident logs to a Coding Agent, it gets confused. The Manager routes *only* relevant data to specific specialists.

In [2]:
def coding_agent(context):
    if "customer_email" in context:
        print("🚨 [Coding Agent] ERROR: Why did I receive a customer email? I am hallucinating a response to the customer!")
        return False
    if "stack_trace" in context:
        print("✅ [Coding Agent] Received isolated stack trace. Generating patch...")
        return True

def support_agent(context):
    if "stack_trace" in context:
        print("🚨 [Support Agent] ERROR: Why did I receive python code? I am hallucinating code to the customer!")
        return False
    if "customer_email" in context:
        print("✅ [Support Agent] Received isolated email. Drafting apology...")
        return True

def manager_agent_router(master_context):
    print("[Manager Agent] Decomposing context and routing to specialists...")
    
    # Isolate contexts
    dev_context = {"stack_trace": master_context["stack_trace"]}
    cx_context = {"customer_email": master_context["customer_email"]}
    
    coding_agent(dev_context)
    support_agent(cx_context)

incident_data = {
    "stack_trace": "ZeroDivisionError: division by zero",
    "customer_email": "My checkout failed! - Alice"
}
manager_agent_router(incident_data)


[Manager Agent] Decomposing context and routing to specialists...
✅ [Coding Agent] Received isolated stack trace. Generating patch...
✅ [Support Agent] Received isolated email. Drafting apology...


---
## Pattern 3: The Escalation Path

Agents must not act as surrogate executives. If they lack data, they must escalate to a human, not invent a resolution.

In [3]:
def analysis_agent(data_feed):
    print("[Analysis Agent] Attempting to find root cause...")
    if not data_feed:
        # Anti-Pattern: Hallucinating a cause
        # return "The server probably ran out of memory."
        
        # Best Practice: Escalation
        print("🚨 [Analysis Agent] ESCALATION TRIGGERED: Data feed is empty. I cannot resolve this.")
        return "ESCALATE_TO_HUMAN"
    
    return "Root cause found."

def manager_agent(data_feed):
    result = analysis_agent(data_feed)
    if result == "ESCALATE_TO_HUMAN":
        print("[Manager Agent] Pausing execution. Paging Human On-Call.")

print("--- Executing with missing data ---")
manager_agent(data_feed=[])


--- Executing with missing data ---
[Analysis Agent] Attempting to find root cause...
🚨 [Analysis Agent] ESCALATION TRIGGERED: Data feed is empty. I cannot resolve this.
[Manager Agent] Pausing execution. Paging Human On-Call.


---
## Pattern 4: Human-in-the-Loop (HITL) Approval

Before executing a high-impact action (like a refund), the agent graph must pause and wait for a human cryptographic signature.

In [4]:
def execute_refund(amount, human_signature):
    if human_signature != "VALID_HUMAN_SIG":
        print("🚨 [System] BLOCKED: Unsigned agent execution attempted.")
        return False
    
    print(f"✅ [System] Verified Human Signature. Refund of ${amount} processed.")
    return True

def agent_workflow():
    print("[Agent] I have determined a $500 refund is required.")
    print("[Agent] PAUSING WORKFLOW. Entering Review Queue...")
    
    # Simulated human review step
    print("\n[Human Approver] Reviewing agent's evidence... Approved.")
    human_sig = "VALID_HUMAN_SIG"
    
    execute_refund(500, human_sig)

agent_workflow()


[Agent] I have determined a $500 refund is required.
[Agent] PAUSING WORKFLOW. Entering Review Queue...

[Human Approver] Reviewing agent's evidence... Approved.
✅ [System] Verified Human Signature. Refund of $500 processed.
